<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Target-module comparison



In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip uninstall torchao

In [3]:
from datasets import load_dataset, Dataset

dataset = load_dataset(
    "yahma/alpaca-cleaned",
    split="train"
)

df = dataset.to_pandas()
df = df.drop_duplicates()

dataset = Dataset.from_pandas(
    df,
    preserve_index=False
)

train_test = dataset.train_test_split(
    test_size=0.20,
    seed=42
)

train_data = train_test["train"]
temp_data = train_test["test"]

validation_test = temp_data.train_test_split(
    test_size=0.50,
    seed=42
)

validation_data = validation_test["train"]
test_data = validation_test["test"]

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Train: 41404
Validation: 5176
Test: 5176


In [4]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


def format_chat(example):
    if example["input"].strip():
        user_content = (
            example["instruction"]
            + "\n\nInput:\n"
            + example["input"]
        )
    else:
        user_content = example["instruction"]

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]}
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }


def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )


validation_test_data = validation_data.select(range(100))

validation_test_data = validation_test_data.map(
    format_chat
)

validation_test_data = validation_test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=validation_test_data.column_names
)


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Setup restored successfully.")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Setup restored successfully.


In [5]:
full_ft_test_data = train_data.select(range(100))

full_ft_test_data = full_ft_test_data.map(
    format_chat
)

full_ft_test_data = full_ft_test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=full_ft_test_data.column_names
)

print("Training examples:", len(full_ft_test_data))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Training examples: 100


In [6]:
!pip install -U "bitsandbytes>=0.46.1"

In [7]:
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("QLoRA base model loaded successfully.")
print("Model dtype:", qlora_base_model.dtype)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

QLoRA base model loaded successfully.
Model dtype: torch.bfloat16


In [8]:
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model
)

qlora_base_model = prepare_model_for_kbit_training(
    qlora_base_model
)

qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

qlora_model = get_peft_model(
    qlora_base_model,
    qlora_config
)

qlora_model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [9]:
import time
import torch
from transformers import TrainingArguments, Trainer

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

qlora_model.config.use_cache = False
qlora_model.train()

training_args = TrainingArguments(
    output_dir="./outputs/qlora_test",
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=full_ft_test_data,
    eval_dataset=validation_test_data,
    data_collator=data_collator
)

start_time = time.time()

train_output = trainer.train()

training_time = time.time() - start_time

peak_memory_gb = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)

eval_output = trainer.evaluate()

eval_loss = eval_output["eval_loss"]

print("\n===== QLoRA Results =====")
print(f"Training time: {training_time:.2f} sec")
print(f"Peak GPU memory: {peak_memory_gb:.2f} GB")
print(f"Train loss: {train_output.training_loss:.4f}")
print(f"Eval loss: {eval_loss:.4f}")

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.384295
10,2.034406
15,1.779441
20,1.963373


Training Loss,Validation Loss,Step
1.963373,1.625969,20



===== QLoRA Results =====
Training time: 12.30 sec
Peak GPU memory: 1.88 GB
Train loss: 2.0404
Eval loss: 1.6260
